# HalluciGuard on Kaggle, part 2: CoVe + CRAG on PopQA, and FActScore biographies

This notebook **reuses the first run** instead of starting over. These files are in the repo under `results/kaggle_run1/`:
- `popqa_gen/generations.jsonl`: Qwen's 987 PopQA eval answers with probe scores (stage 1)
- `popqa_dev/generations.jsonl`: the dev split, used to recompute the probe threshold (0.742510)
- `popqa/blackboard/`, `popqa/pipeline/`: the finished stage-2 results for these two systems

**What still has to run:**
1. Rebuild the PopQA Wikipedia KB (CPU and internet only). It isn't in the repo.
2. Stage 1 for the **biographies only** (Qwen on the GPU, 100 bios). No biography generations were saved.
3. **CoVe and CRAG** on the same PopQA answers, then compare all four systems.
4. **FActScore biographies**: mitigation with all four systems, then scoring by the Gemma-2-9B judge.

**Notebook settings:** Accelerator = **GPU T4 x2**, Internet = **on**. Add your probe `.joblib` dataset as before.

**If the session restarts:** run cells 0, 1 and 2 (fast: they only copy files and recompute the threshold),
then skip to the cell you were on. Every stage-2 run resumes from its checkpoint, and finished files are not rebuilt.

In [ ]:
# 0. code + dependencies
%cd /kaggle/working
import os
if not os.path.isdir("HalluciGuard"):
    !git clone -q -b feature/real-blackboard https://github.com/Yaswanth8390/HalluciGuard.git
%cd /kaggle/working/HalluciGuard
!git pull -q origin feature/real-blackboard
!pip install -q -r requirements-kaggle.txt
!git log --oneline -3

## 1. Settings

In [ ]:
N_POPQA, N_DEV, N_BIO = 999, 300, 100   # must match the first run (the KB is rebuilt for exactly these questions)
BUDGET_QA = 8          # LLM calls per PopQA question for CoVe / CRAG (same as the blackboard run)
BUDGET_BIO = 24        # LLM calls per biography
WORKERS = 8            # parallel items = 2 GPUs x 2 models x 2 parallel requests per model
PROBE = ""             # empty = auto-detect the .joblib under /kaggle/input

import glob, json, os, shutil, time
os.makedirs("runs", exist_ok=True)
if not PROBE:
    found = sorted(glob.glob("/kaggle/input/**/*.joblib", recursive=True))
    PROBE = found[0] if found else ""
assert PROBE, "add your probe .joblib as a Kaggle dataset (the first run used it; the threshold depends on it)"
probe_arg = f"--probe {PROBE}"
import joblib
_b = joblib.load(PROBE)
print("probe:", PROBE, {k: _b[k] for k in ("feature_dim", "n_train", "test_metrics") if k in _b})
GUARD = "" if _b.get("train_questions") else "--allow_train_overlap"

## 2. Import the first run's results
The files are copied into the `runs/` layout that the scripts expect, then checked: the eval answers must
line up with the blackboard and pipeline results row for row, and the dev split must give the same threshold.

In [ ]:
# stage-1 answers from your Kaggle dataset (eval = 987 rows, dev = 296 rows; told apart by size, not name)
KAGGLE_GEN = ["/kaggle/input/datasets/yaswanthreddy122312/qwen-blackboard/Qwen_rename.jsonl",
              "/kaggle/input/datasets/yaswanthreddy122312/qwen-blackboard/Thresh.jsonl"]
def _nrows(p):
    return sum(1 for l in open(p, encoding="utf-8") if l.strip())
_sizes = {p: _nrows(p) for p in KAGGLE_GEN}
print("dataset files:", _sizes)
_eval = [p for p, n in _sizes.items() if n == 987]
_dev = [p for p, n in _sizes.items() if n == 296]
assert len(_eval) == 1 and len(_dev) == 1, f"expected one 987-row (eval) and one 296-row (dev) file, got {_sizes}"
for src_, dst_ in [(_eval[0], "runs/popqa_gen/generations.jsonl"), (_dev[0], "runs/popqa_dev/generations.jsonl")]:
    os.makedirs(os.path.dirname(dst_), exist_ok=True)
    shutil.copy(src_, dst_)
    print(f"{src_} -> {dst_}")

# blackboard + pipeline results of the first run (in the repo)
SRC = "results/kaggle_run1"
for rel in ["popqa/blackboard/per_item.csv", "popqa/blackboard/metrics.json", "popqa/blackboard/summary.md",
            "popqa/pipeline/per_item.csv", "popqa/pipeline/metrics.json", "popqa/pipeline/summary.md"]:
    dst = os.path.join("runs", rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        shutil.copy(os.path.join(SRC, rel), dst)

import csv
GEN = [json.loads(l) for l in open("runs/popqa_gen/generations.jsonl", encoding="utf-8") if l.strip()]
assert len(GEN) == 987 and all(r.get("prob_hallucinated") not in (None, "") for r in GEN), "unexpected generations file"
for s_ in ["blackboard", "pipeline"]:
    rows_ = list(csv.DictReader(open(f"runs/popqa/{s_}/per_item.csv", encoding="utf-8")))
    assert [r["question"] for r in rows_] == [g["question"] for g in GEN], f"{s_} results do not match the answers"
print(f"eval answers: {len(GEN)} (groups: {sorted(set(g['group'] for g in GEN))}); blackboard + pipeline results match")

!python choose_threshold.py --dev runs/popqa_dev/generations.jsonl --out runs/popqa_dev/threshold.json | tail -1
THRESH = json.load(open("runs/popqa_dev/threshold.json"))["threshold"]
_first = json.load(open("runs/popqa/blackboard/metrics.json"))["config"]["threshold"]
assert abs(THRESH - _first) < 1e-9, f"threshold {THRESH} differs from the first run's {_first}"
print("risk threshold (same as the first run):", THRESH)

## 3. Data: rebuild the PopQA Wikipedia KB, and fetch the FActScore biographies
The question sample is seeded, so `prepare_eval_data.py` picks the same 999 questions as the first run.
The cell checks this against the saved answers. It also checks that the gold passages match the ones the
first run had, which tells us the Wikipedia articles haven't changed much since then. Wikipedia rate-limits
Kaggle IPs, so the cell repeats until no articles are missing. Each round fetches only what is still missing.

In [ ]:
def wiki_missing():
    have = set()
    if os.path.exists("eval_data/wiki_cache.jsonl"):
        for l in open("eval_data/wiki_cache.jsonl", encoding="utf-8"):
            r = json.loads(l) if l.strip() else {}
            if r.get("text") or r.get("missing"):
                have.add(r["title"])
    titles = {json.loads(l).get("s_wiki_title") for l in open("eval_data/popqa.jsonl", encoding="utf-8") if l.strip()}
    return sorted(t for t in titles if t and t not in have)

for round_ in range(4):
    !python prepare_eval_data.py --dataset popqa --n {N_POPQA} --dev {N_DEV} --out eval_data/popqa.jsonl --wiki
    missing_ = wiki_missing()
    print(f"articles still missing after round {round_ + 1}: {len(missing_)}", missing_[:10])
    if not missing_:
        break
    time.sleep(120)

# same questions as the first run? (a stage-1 row's "id" is its index in eval_data/popqa.jsonl)
DATA = [json.loads(l) for l in open("eval_data/popqa.jsonl", encoding="utf-8") if l.strip()]
bad_q = [g["id"] for g in GEN if DATA[int(g["id"])]["question"] != g["question"]]
assert not bad_q, f"{len(bad_q)} questions differ from the first run: the KB would not match the answers"
same_kb = sum(DATA[int(g["id"])].get("knowledge", "") == g.get("knowledge", "") for g in GEN)
print(f"questions match the first run: 987/987; gold passage unchanged for {same_kb}/987 "
      f"({same_kb / 987:.1%}; above ~95% means the KB is effectively the one blackboard/pipeline used)")

if not os.path.exists("eval_data/bio_kb.jsonl"):
    !python prepare_factscore.py --n {N_BIO} --out eval_data/bio.jsonl
!wc -l eval_data/popqa_kb.jsonl eval_data/bio.jsonl eval_data/bio_kb.jsonl

## 4. Stage 1 for the biographies only (Qwen2.5-7B on the GPU)
This must run **before** the Ollama agents start, because Qwen needs the GPUs. It exits and frees them when done.
PopQA stage 1 is **not** rerun.

In [ ]:
if not os.path.exists("runs/bio/generations.jsonl"):
    !python eval_bio.py --dataset eval_data/bio.jsonl --generate_only --batch_size 8 {probe_arg} --out_dir runs/bio
!wc -l runs/bio/generations.jsonl
!nvidia-smi --query-gpu=memory.used --format=csv

## 5. Start the agent models (Ollama, both T4s) and map them to roles
Same agents as the first run. CoVe and CRAG use one model (Llama-3.1-8B, `LLM_BASELINE_MODEL`), as in their papers.
The script checks that every model runs `100% GPU` and restarts a crashed server.

In [ ]:
!bash scripts/kaggle_serve_agents.sh

In [ ]:
AGENT_ENV = {
    "LLM_BASE_URL": "http://localhost:11434/v1,http://localhost:11435/v1",   # GPU0, GPU1
    "LLM_API_KEY": "ollama",
    "GROQ_MODEL": "llama3.1:8b",
    "LLM_JSON_MODE": "1",
    "LLM_TIMEOUT": "180",
    "LLM_MAX_TOKENS": "512",
    "LLM_VERIFIER_MODEL": "llama3.1:8b", "LLM_HYPOTHESIS_MODEL": "llama3.1:8b", "LLM_ANSWER_MODEL": "llama3.1:8b",
    "LLM_VERIFIER2_MODEL": "mistral:7b",
    "LLM_DECOMPOSER_MODEL": "mistral:7b", "LLM_REFORMULATOR_MODEL": "mistral:7b", "LLM_COMPOSER_MODEL": "mistral:7b",
    "LLM_CORRECTOR_MODEL": "mistral:7b", "LLM_EXTRACTOR_MODEL": "mistral:7b", "LLM_ABSTENTION_MODEL": "mistral:7b",
    "LLM_CONFLICT_MODEL": "mistral:7b",
    "LLM_BASELINE_MODEL": "llama3.1:8b",    # CoVe / CRAG
    "LLM_JUDGE_MODEL": "gemma2:9b",         # FActScore judge, never an agent
}
os.environ.update(AGENT_ENV)
COMMON = (f"--answers_file runs/popqa_gen/generations.jsonl --kb_file eval_data/popqa_kb.jsonl --threshold {THRESH} {GUARD} {probe_arg} "
          f"--isolate_memory --no_llm_abstention --workers {WORKERS} --retries 3 --fallback_on_error")

### 5a. Smoke test (10 questions per baseline)
This catches setup problems in a few minutes instead of hours in. Look for sensible `with blackboard` numbers and no errors.

In [ ]:
for system in ["cove", "crag"]:
    !python eval_pipeline.py {COMMON} --limit 10 --system {system} --max_llm_calls {BUDGET_QA} --out_dir runs/smoke_{system}
    !grep -E "counts|API errors|LLM calls" runs/smoke_{system}/summary.md
!bash scripts/kaggle_serve_agents.sh status

## 6. PopQA: CoVe and CRAG on the same 987 answers
These use the same probe gate (threshold), KB, budget and settings as the blackboard run, so the four systems
are paired item by item. Each run takes a few hours. A stopped run resumes from `runs/popqa/<system>/bb_partial.jsonl`.

In [ ]:
for system in ["cove", "crag"]:
    !python eval_pipeline.py {COMMON} --system {system} --max_llm_calls {BUDGET_QA} --out_dir runs/popqa/{system}
    !bash scripts/kaggle_serve_agents.sh status

In [ ]:
# paired comparison of all four systems (blackboard + pipeline come from the first run)
!python compare_systems.py --runs blackboard=runs/popqa/blackboard pipeline=runs/popqa/pipeline cove=runs/popqa/cove crag=runs/popqa/crag --reference pipeline --out runs/popqa/comparison.md
!cat runs/popqa/comparison.md

In [ ]:
# same comparison with the blackboard as the reference (is it better than each published baseline?)
!python compare_systems.py --runs blackboard=runs/popqa/blackboard pipeline=runs/popqa/pipeline cove=runs/popqa/cove crag=runs/popqa/crag --reference blackboard --out runs/popqa/comparison_vs_blackboard.md
!cat runs/popqa/comparison_vs_blackboard.md

## 7. FActScore biographies (long-form): mitigation, then scoring
Every biography is mitigated (`--threshold 0`), because the probe was trained on one-word answers. Its score
is still recorded, and the summary reports how well it predicts a low FActScore.

7a runs all four systems with the agents loaded. 7b then swaps the agents for the Gemma-2-9B judge on both
GPUs and scores the biographies. Mitigated rows are reused, not rerun.

In [ ]:
# 7a. mitigation (scoring skipped)
os.environ["LLM_MAX_TOKENS"] = "1024"   # claim lists and rewritten biographies are long
!python eval_bio.py --answers_file runs/bio/generations.jsonl --kb eval_data/bio_kb.jsonl --systems blackboard,pipeline,cove,crag --max_llm_calls {BUDGET_BIO} --workers {WORKERS} --out_dir runs/bio --skip_scoring

In [ ]:
# 7b. judge on both GPUs, then score
os.environ["LLM_MAX_TOKENS"] = "1024"
!bash scripts/kaggle_serve_agents.sh judge
!python eval_bio.py --answers_file runs/bio/generations.jsonl --kb eval_data/bio_kb.jsonl --systems blackboard,pipeline,cove,crag --max_llm_calls {BUDGET_BIO} --workers {WORKERS} --out_dir runs/bio
!bash scripts/kaggle_serve_agents.sh agents
!cat runs/bio/summary.md

### 7c. Validate the judge (optional, by hand after the run)
This exports 200 judged facts with their Wikipedia context and hides the judge's label. Fill `human_label_1`
(1 = supported, 0 = not) in `runs/bio/annotation/annotate.csv`, then run `judge_agreement.py score` on any machine.

In [ ]:
!python judge_agreement.py export --bio_dir runs/bio --answers_file runs/bio/generations.jsonl --kb eval_data/bio_kb.jsonl --n 200 --out runs/bio/annotation

## 8. Results (download `results.zip` from the output panel)

In [ ]:
!cat runs/popqa/comparison.md; echo; cat runs/popqa/comparison_vs_blackboard.md; echo; cat runs/bio/summary.md
!cd /kaggle/working/HalluciGuard && zip -qr /kaggle/working/results.zip runs eval_data/popqa*.jsonl eval_data/bio.jsonl -x "eval_data/*_kb.jsonl" && ls -lh /kaggle/working/results.zip